# 🤖 Project 03: AI 策略生成器 — Demo

這個 notebook 展示如何用自然語言自動生成交易策略、回測驗證、並進行參數優化。

In [ ]:
import sys
sys.path.insert(0, 'projects/03-ai-strategy-generator/src')

# 1. 自然語言策略生成
from strategy_agent import StrategyAgent

agent = StrategyAgent()
result = agent.generate(
    '幫我寫一個均值回歸策略，用布林通道和 RSI 結合，'
    '在下軌且 RSI 超賣時買入，到上軌且 RSI 超買時賣出'
)

print(f'Strategy Name: {result.strategy_name}')
print(f'Template: {result.template_used}')
print(f'Parameters: {result.parameters}')

In [ ]:
# 2. 查看生成的策略代碼
print('=== Generated Strategy Code ===')
print(result.code[:1000])  # 顯示前 1000 字元

In [ ]:
# 3. 代碼驗證
from code_validator import validate_strategy_code

validation = validate_strategy_code(result.code)
print(f'Valid: {validation.is_valid}')
if validation.errors:
    print(f'Errors: {validation.errors}')
else:
    print('✅ Code passed all safety checks')

In [ ]:
# 4. 執行回測
from backtester import run_backtest

backtest_result = run_backtest(result.code)
print(f'Total Return: {backtest_result.total_return:.2%}')
print(f'Sharpe Ratio: {backtest_result.sharpe_ratio:.2f}')
print(f'Max Drawdown: {backtest_result.max_drawdown:.2%}')
print(f'Win Rate: {backtest_result.win_rate:.2%}')

In [ ]:
# 5. 參數優化
from optimizer import grid_search, walk_forward_validate

# 網格搜索
best_params, best_score = grid_search(
    result.code,
    param_grid={
        'bb_period': [15, 20, 25],
        'bb_std': [1.5, 2.0, 2.5],
        'rsi_period': [10, 14, 20],
        'rsi_oversold': [25, 30, 35],
        'rsi_overbought': [65, 70, 75],
    },
    metric='sharpe_ratio'
)

print(f'Best Parameters: {best_params}')
print(f'Best Sharpe Ratio: {best_score:.2f}')

# Walk-Forward 驗證
wf_result = walk_forward_validate(result.code, best_params, n_splits=5)
print(f'\nWalk-Forward Results:')
print(f'  Mean Return: {wf_result.mean_return:.2%}')
print(f'  Std Return: {wf_result.std_return:.2%}')
print(f'  Overfit Score: {wf_result.overfit_score:.2f}')

## 🎯 Summary

這個核心項目展示了：
- **NL→策略**: 自然語言描述 → 自動生成 Backtrader 代碼
- **代碼驗證**: AST 分析 + 安全沙箱檢查
- **回測執行**: 一鍵生成回測報告
- **參數優化**: 網格搜索 + Walk-Forward 驗證防過擬合